## Templates

Das Template ist bewusst auf einen einfachen Platzhalter reduziert. `template.yaml` beschreibt den Ablauf und die Eingabefelder des Backstage-Templates, während der Ordner `content` die Dateien enthält, die beim Ausführen in das neue Repository kopiert werden.

Die `README.md` dient dabei als minimaler Beispielinhalt und kann später schrittweise durch die eigentliche Lemon-Shop-Applikation, Katalogdateien und Kubernetes-Konfigurationen ersetzt werden.

    ~/mybackstage/template/
    ├── template.yaml
    └── content/
        ├── README.md

In [ ]:
%%bash
rm -f ~/mybackstage/examples/template/content/*
cat > ~/mybackstage/examples/template/template.yaml <<'EOF'
apiVersion: scaffolder.backstage.io/v1beta3
kind: Template
metadata:
  name: lemon-shop
  namespace: default
  title: Lemon Shop
  description: Erstellt ein neues Repository für den Lemon Shop
  tags:
    - lemon-shop
    - github
spec:
  owner: user:default/guest
  type: service

  parameters:
    - title: Repository
      required:
        - repoUrl
      properties:
        repoUrl:
          title: GitHub Repository
          type: string
          description: Ziel-Repository auf GitHub
          ui:field: RepoUrlPicker
          ui:options:
            allowedHosts:
              - github.com
            allowedOwners:
              - marcel-cli

  steps:
    - id: fetch
      name: Inhalt vorbereiten
      action: fetch:template
      input:
        url: ./content
        values:
          name: ${{ parameters.repoUrl | parseRepoUrl | pick('repo') }}

    - id: publish
      name: Repository veröffentlichen
      action: publish:github
      input:
        repoUrl: ${{ parameters.repoUrl }}
        repoVisibility: public
        defaultBranch: main

  output:
    links:
      - title: GitHub Repository
        url: ${{ steps.publish.output.remoteUrl }}
EOF

### Template Dateien

In [ ]:
%%bash
cat > ~/mybackstage/examples/template/content/README.md <<'EOF'
# ${{ values.systemTitle }}

${{ values.repositoryDescription }}

Dieses Repository ist ein Platzhalter für den Lemon Shop.

## Bestandteile

- System: `${{ values.systemName }}`
- Component: `shop-frontend`
- Component: `product-service`
- Component: `order-service`
- Component: `payment-service`
- Resource: `database`
- Resource: `kubernetes`

EOF